# 08.5 — Model Serving: Quantization, vLLM, llama.cpp

**The production problem:** A 7B parameter model in fp32 = 28GB. Running it requires expensive hardware and is slow. Going from research to production requires understanding:
1. **Quantization:** Reduce precision to save memory and speed up inference
2. **Batching:** Process multiple requests on the same GPU pass
3. **KV-Cache:** Avoid recomputing attention for the same prefix
4. **Inference engines:** vLLM, llama.cpp — what they optimize and when to use each

---

In [ ]:
import numpy as np
import struct

# ---- Quantization from scratch ----
# Understanding what int8 quantization actually does.

def quantize_int8(weights: np.ndarray) -> tuple:
    """
    Absmax INT8 quantization.
    Maps float32 weights to int8 [-127, 127].
    
    scale = max(|w|) / 127
    q = round(w / scale)
    w_approx = q * scale  (dequantize)
    """
    scale = np.abs(weights).max() / 127.0
    q = np.round(weights / scale).astype(np.int8)
    return q, scale


def dequantize_int8(q: np.ndarray, scale: float) -> np.ndarray:
    return q.astype(np.float32) * scale


def quantize_int4(weights: np.ndarray) -> tuple:
    """
    INT4 quantization: maps to [-7, 7] (4-bit signed).
    Stored as pairs in uint8 (two 4-bit values per byte).
    """
    scale = np.abs(weights).max() / 7.0
    q = np.clip(np.round(weights / scale), -7, 7).astype(np.int8)
    return q, scale


# Simulate a weight matrix (like a transformer linear layer)
np.random.seed(42)
W = np.random.randn(256, 256).astype(np.float32) * 0.02  # small weights, typical for transformers

# Quantize
q8, scale8 = quantize_int8(W)
q4, scale4 = quantize_int4(W)

# Dequantize and measure error
W_int8 = dequantize_int8(q8, scale8)
W_int4 = dequantize_int8(q4, scale4)

err8 = np.abs(W - W_int8)
err4 = np.abs(W - W_int4)

print("Quantization error (float32 vs quantized):")
print(f"  INT8: mean_abs_err={err8.mean():.6f}  max_err={err8.max():.6f}  relative={err8.mean()/np.abs(W).mean():.4f}")
print(f"  INT4: mean_abs_err={err4.mean():.6f}  max_err={err4.max():.6f}  relative={err4.mean()/np.abs(W).mean():.4f}")

print()
print("Memory comparison for a 7B model:")
params = 7_000_000_000
print(f"  fp32:  {params * 4 / 1e9:.1f} GB")
print(f"  fp16:  {params * 2 / 1e9:.1f} GB  (halved, near-zero quality loss)")
print(f"  int8:  {params * 1 / 1e9:.1f} GB  (quartered, ~1% quality loss on benchmarks)")
print(f"  int4:  {params * 0.5 / 1e9:.1f} GB  (8× smaller, 2-4% quality loss)")
print(f"  nf4:   {params * 0.5 / 1e9:.1f} GB  (same size, better quality than int4 — used in QLoRA)")

In [ ]:
# ---- Per-channel vs absmax quantization ----
# Absmax uses one scale per tensor (lossy for outlier-heavy tensors).
# Per-channel uses one scale per row/column (better quality).

def quantize_per_channel(W: np.ndarray) -> tuple:
    """
    Per-row quantization: each output channel gets its own scale.
    More accurate than absmax for tensors with outlier neurons.
    """
    scales = np.abs(W).max(axis=1, keepdims=True) / 127.0  # [rows, 1]
    scales = np.maximum(scales, 1e-8)  # avoid div by zero
    q = np.round(W / scales).astype(np.int8)
    return q, scales


# Create a matrix with outliers (common in LLMs — some attention heads have large weights)
W_outlier = np.random.randn(32, 32).astype(np.float32) * 0.01
W_outlier[5, :] *= 100   # Row 5 has 100× larger values
W_outlier[15, :] *= 50   # Row 15 has 50× larger values

q_abs, scale_abs = quantize_int8(W_outlier)
W_abs = dequantize_int8(q_abs, scale_abs)
err_abs = np.abs(W_outlier - W_abs)

q_chan, scales_chan = quantize_per_channel(W_outlier)
W_chan = (q_chan.astype(np.float32) * scales_chan)
err_chan = np.abs(W_outlier - W_chan)

print("Quantization quality with outlier neurons:")
print(f"  Absmax:       mean_err={err_abs.mean():.4f}  (outlier row dominates scale)")
print(f"  Per-channel:  mean_err={err_chan.mean():.6f}  (each row has its own scale)")
print()
print("LLM.int8() (bitsandbytes):")
print("  Decompose matrix: outlier dims → fp16, non-outlier → int8")
print("  Only ~0.1% of values are outliers but dominate absmax scale")
print("  This hybrid approach enables zero-quality-loss int8 at 7B+ scale")

print()
print("NF4 (NormalFloat4 from QLoRA):")
print("  Weights are approximately normally distributed")
print("  NF4 places quantization levels at equal-probability intervals of N(0,1)")
print("  → Lower quantization error than uniform int4 for neural network weights")

In [ ]:
# ---- KV-Cache: the key to fast autoregressive generation ----

print("=== KV-Cache ===")
print()
print("The generation problem:")
print("  At step t, we have tokens [t0, t1, ..., t_{t-1}] and predict t_t.")
print("  Without cache: recompute K and V for ALL previous tokens at each step.")
print("  Cost: O(T^2) for T tokens generated.")
print()
print("With KV-cache:")
print("  Compute K_t and V_t for the new token only.")
print("  Append to cached K and V from previous steps.")
print("  Attention: Q_t (new) attends to all cached K (O(T) per step, O(T^2) total but no wasted recompute).")
print()

def simulate_generation_without_cache(seq_len, d_model, n_heads, n_layers):
    """FLOPs for autoregressive generation without KV cache."""
    flops_per_step = []
    for t in range(1, seq_len + 1):
        # At step t: full self-attention over t tokens, n_layers
        attn_flops = n_layers * (t * t * d_model)  # O(t^2 * d)
        flops_per_step.append(attn_flops)
    return sum(flops_per_step)


def simulate_generation_with_cache(seq_len, d_model, n_heads, n_layers):
    """FLOPs for autoregressive generation with KV cache."""
    flops_per_step = []
    for t in range(1, seq_len + 1):
        # At step t: only compute Q_t, attend to cached K_{1..t}
        attn_flops = n_layers * (t * d_model)  # O(t * d) per step
        flops_per_step.append(attn_flops)
    return sum(flops_per_step)


d, h, L = 768, 12, 12
for T in [64, 256, 1024, 4096]:
    no_cache = simulate_generation_without_cache(T, d, h, L)
    with_cache = simulate_generation_with_cache(T, d, h, L)
    speedup = no_cache / with_cache
    print(f"  T={T:5d}: no_cache={no_cache/1e9:.1f}G FLOPs, with_cache={with_cache/1e9:.1f}G FLOPs, speedup={speedup:.1f}×")

print()
print("KV cache memory cost (per token, per layer, float16):")
n_layers = 32  # LLaMA-7B
d_head = 128   # 4096 / 32 heads
n_kv_heads = 32
bytes_per_token = n_layers * 2 * n_kv_heads * d_head * 2  # K + V, float16
print(f"  LLaMA-7B: {bytes_per_token / 1024:.1f} KB per token")
print(f"  For 2048 context: {bytes_per_token * 2048 / 1024**2:.1f} MB")
print(f"  For 32K context:  {bytes_per_token * 32768 / 1024**2:.1f} MB")

In [ ]:
# ---- vLLM: PagedAttention ----
# The key insight in vLLM (Kwon et al., 2023):
# KV cache fragmentation wastes 20-40% of GPU memory.
# PagedAttention manages KV cache like OS page tables.

print("=== vLLM: PagedAttention ===")
print()
print("Problem with naive KV cache:")
print("  Pre-allocate max context length (e.g., 2048 tokens) per request.")
print("  Request ends at token 50 → 1998 tokens of KV cache wasted (reserved but unused).")
print("  Internal fragmentation wastes 20-40% of GPU memory.")
print()
print("PagedAttention solution:")
print("  Allocate KV cache in fixed-size 'blocks' (like OS memory pages).")
print("  Blocks can be non-contiguous in GPU memory (use block table for indirection).")
print("  Only allocate blocks as tokens are generated — no reservation.")
print("  Result: near-zero fragmentation → 2-4× more requests in flight.")
print()
print("vLLM throughput gains:")
print("  vs naive serving: 24× higher throughput")
print("  vs HuggingFace text-generation-inference: 2-4×")
print()

print("=== Inference engine comparison ===")
print()

engines = [
    ("vLLM",
     "Python, GPU only",
     "Best GPU throughput. PagedAttention. OpenAI-compatible API.",
     "GPU server, production API, high concurrent load"),
    
    ("llama.cpp",
     "C++, CPU/GPU/Metal",
     "Runs on CPU! GGUF format, GGML quantization. Minimal dependencies.",
     "Local inference, CPU-only machines, edge deployment"),
    
    ("HuggingFace TGI",
     "Python, GPU",
     "Continuous batching, tensor parallelism, Paged KV cache.",
     "Production, managed service (Inference Endpoints)"),
    
    ("ONNX Runtime",
     "Python/C++, CPU/GPU",
     "Cross-platform, hardware vendor optimizations.",
     "Encoder models (BERT), embedding models"),
    
    ("TensorRT-LLM",
     "C++/Python, NVIDIA GPU only",
     "Maximum NVIDIA GPU utilization, complex setup.",
     "Production on NVIDIA hardware, lowest latency"),
]

for name, platform, strength, use_case in engines:
    print(f"  {name:20} [{platform}]")
    print(f"  {'':20} + {strength}")
    print(f"  {'':20} Use when: {use_case}")
    print()

In [ ]:
# ---- Continuous batching: the key to throughput ----

print("=== Continuous Batching ===")
print()
print("Static batching (naive):")
print("  Wait until batch is full, then process all together.")
print("  Problem: short requests finish early but GPU sits idle waiting for long ones.")
print("  GPU utilization: 20-40%")
print()
print("Continuous batching (vLLM, TGI):")
print("  When a request finishes, immediately insert a new one.")
print("  Batch size varies token-by-token, not request-by-request.")
print("  GPU utilization: 70-90%")
print()

# Simulate: compare static vs continuous batching throughput
import time

def simulate_static_batching(requests, batch_size, gpu_step_time_ms=10):
    """Process requests in fixed batches."""
    total_time = 0
    for i in range(0, len(requests), batch_size):
        batch = requests[i:i+batch_size]
        # Must wait for the longest request in the batch
        max_tokens = max(r for r in batch)
        total_time += max_tokens * gpu_step_time_ms
    return total_time


def simulate_continuous_batching(requests, batch_size, gpu_step_time_ms=10):
    """Continuous batching: replace finished requests immediately."""
    queue = list(requests)
    active = [queue.pop(0) for _ in range(min(batch_size, len(queue)))]
    total_time = 0
    
    while active:
        # One GPU step
        total_time += gpu_step_time_ms
        # Decrement all active requests
        active = [r - 1 for r in active]
        # Remove finished, add new ones
        active = [r for r in active if r > 0]
        while len(active) < batch_size and queue:
            active.append(queue.pop(0))
    
    return total_time


np.random.seed(42)
# Mix of short (50 tokens) and long (500 tokens) requests
requests = list(np.random.choice([50, 100, 200, 500], size=100, p=[0.5, 0.3, 0.15, 0.05]))
batch_size = 8

t_static = simulate_static_batching(requests, batch_size)
t_continuous = simulate_continuous_batching(requests, batch_size)

print(f"100 requests (mix of 50-500 tokens), batch_size={batch_size}:")
print(f"  Static batching:     {t_static:,} ms")
print(f"  Continuous batching: {t_continuous:,} ms")
print(f"  Throughput gain:     {t_static/t_continuous:.1f}×")

In [ ]:
# ---- Quick-reference: deployment decision tree ----

print("=== Deployment Decision Guide ===")
print()
print("Model size:")
print("  < 1B params:   Any hardware. Even CPU inference is OK.")
print("  1B-7B:         Single GPU (RTX 3090/4090 24GB) with fp16 or int4")
print("  7B-13B:        A100 40GB (fp16) or 2× 24GB consumer GPUs")
print("  13B-70B:       A100 80GB or multi-GPU with tensor parallelism")
print("  70B+:          Multi-node or API (Claude, GPT-4, Gemini)")
print()
print("Use case:")
print("  Prototype/dev:   HuggingFace transformers + pipeline()")
print("  Local/CPU:       llama.cpp (GGUF format, Q4_K_M quantization)")
print("  GPU production:  vLLM (OpenAI-compatible, continuous batching)")
print("  Embeddings only: sentence-transformers + ONNX Runtime")
print("  Cloud API:       Anthropic/OpenAI API (no infra management)")
print()
print("Quantization guide:")
print("  fp16:    Always use for inference. Free speedup on modern GPUs.")
print("  int8:    Use when memory constrained. bitsandbytes LLM.int8().")
print("  int4:    GGUF Q4_K_M (llama.cpp) or GPTQ. Good quality/size.")
print("  nf4:     QLoRA fine-tuning only. Not for standalone serving.")
print()
print("The 80/20:")
print("  1. Use the API (Claude/GPT) until you have real scale reasons not to.")
print("  2. When you do self-host: vLLM + int4 quantization is usually the right starting point.")
print("  3. Only optimize further after profiling real traffic patterns.")

## Summary — Module 08 Complete

You've covered the full production NLP stack:
1. **Classification metrics:** When accuracy lies, use F1/AUC/PR curves
2. **Generation metrics:** BLEU for precision, ROUGE for recall, BERTScore for semantics
3. **LLM-as-judge:** Pairwise > pointwise; swap positions; explicit rubrics
4. **Latency profiling:** P95/P99 > mean; LLM generation is always the bottleneck
5. **Model serving:** Quantization, KV-cache, continuous batching, vLLM vs llama.cpp

---

## 🎓 Curriculum Complete

You've built the entire NLP engineering stack from scratch:

| Module | Topics |
|--------|--------|
| 01 Text Fundamentals | Unicode, Regex, BPE, Stemming, Sentence Segmentation |
| 02 Classical Retrieval | Inverted Index, TF-IDF, BM25, Query Expansion, Evaluation |
| 03 Classical ML | Naive Bayes, Logistic Regression, N-grams, Edit Distance, CRF |
| 04 Word Vectors | PMI/PPMI, Word2Vec, GloVe, FastText, Analogy Evaluation |
| 05 Sequence Models | RNN, LSTM, GRU, Seq2Seq, Beam Search |
| 06 Attention & Transformers | Bahdanau, Scaled Dot-Product, Multi-Head, PE, Encoder, Decoder, Mini-GPT |
| 07 Modern NLP | BERT, Fine-tuning, Sentence Embeddings+FAISS, DPR, RAG, LoRA, Prompting |
| 08 Evaluation & Production | Classification Metrics, Generation Metrics, LLM Judge, Profiling, Serving |

**What to do next:** Pick a real project. Build something that uses the full pipeline end-to-end — a search engine, a QA system, a fine-tuned classifier. The best way to solidify this is to apply it.